# 02. GNINA 설치 + 도킹 속도 측정 (CDK2 / PDB 3RPR)

**실행 전:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후, 위에서부터 셀을 순서대로 실행.

| 셀 | 내용 | 예상 시간 |
|---|---|---|
| 1 | Google Drive 연결, 폴더 생성 | 10초 |
| 2 | 패키지 설치 (RDKit, OpenBabel) | 1분 |
| 3 | GNINA 설치 (Drive에 저장 → 다음부터는 복사만) | 처음 3~5분 |
| 4 | 3RPR 수용체 준비 (물·리간드 제거, 수소 추가) | 10초 |
| 5 | 리간드 20개 3D 구조 생성 | 30초 |
| 6 | 도킹 속도 측정 (A: 기본 설정 20개 / B: CNN 끔 5개) | 5~15분 |
| 7 | 3,000개 예상 시간 계산 + skill 문서용 요약 저장 | 즉시 |

결과는 모두 `내 드라이브/cdk2/experiment/02_gnina_speed/` 에 저장됨.

In [ ]:
# 셀 1: Drive 연결
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/cdk2'
OUT  = f'{BASE}/experiment/02_gnina_speed'
TOOLS = f'{BASE}/tools'
for d in [OUT, TOOLS, f'{OUT}/ligands', f'{OUT}/docked']:
    os.makedirs(d, exist_ok=True)
print('저장 위치:', OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 셀 2: 패키지 설치
!pip -q install rdkit openbabel-wheel
import rdkit; print('RDKit', rdkit.__version__)

In [ ]:
# 셀 3: GNINA 설치
# Drive에 저장된 바이너리가 있으면 복사만, 없으면 GitHub 최신 릴리스에서 받아 작동하는 파일을 고름.
import os, glob, json, shutil, subprocess, urllib.request, site

# Colab에 깔린 CUDA/cuDNN 라이브러리를 GNINA가 찾을 수 있게 경로 추가
libdirs = []
for sp in site.getsitepackages():
    libdirs += glob.glob(f'{sp}/nvidia/*/lib')
libdirs += ['/usr/local/cuda/lib64', '/usr/lib64-nvidia']
os.environ['LD_LIBRARY_PATH'] = ':'.join(libdirs + [os.environ.get('LD_LIBRARY_PATH', '')])

GNINA = '/content/gnina'

def works(path):
    try:
        r = subprocess.run([path, '--version'], capture_output=True, text=True, timeout=120)
        txt = (r.stdout + r.stderr).strip()
        return r.returncode == 0 and 'gnina' in txt.lower(), txt[:300]
    except Exception as e:
        return False, str(e)[:300]

saved = f'{TOOLS}/gnina'
ok = False
if os.path.exists(saved):
    shutil.copy(saved, GNINA); os.chmod(GNINA, 0o755)
    ok, msg = works(GNINA)
    print('Drive에 저장된 GNINA 사용:', ok, msg)

if not ok:
    rels = json.load(urllib.request.urlopen('https://api.github.com/repos/gnina/gnina/releases?per_page=6'))
    cands = []
    for r in rels:
        for a in r['assets']:
            if a['name'].startswith('gnina') and a['size'] > 50e6:
                cands.append((r['tag_name'], a['name'], a['browser_download_url'], a['size']))
    # CUDA 12 빌드 우선 (Colab은 CUDA 12)
    cands.sort(key=lambda c: (0 if 'cuda12' in c[1] else 1))
    for tag, name, url, size in cands:
        print(f'시도: {tag} / {name} ({size/1e6:.0f} MB) ...')
        subprocess.run(['wget', '-q', '-O', GNINA, url])
        os.chmod(GNINA, 0o755)
        ok, msg = works(GNINA)
        print('  →', '성공' if ok else '실패', msg.splitlines()[0] if msg else '')
        if ok:
            shutil.copy(GNINA, saved)   # 다음 세션부터는 다운로드 생략
            GNINA_VERSION = f'{tag} ({name})'
            break

assert ok, 'GNINA 설치 실패 — 이 셀의 출력 전체를 복사해서 Claude에게 보내줘'
GNINA_VERSION = subprocess.run([GNINA, '--version'], capture_output=True, text=True).stdout.strip().splitlines()[0]
print('\n설치 완료:', GNINA_VERSION)

In [ ]:
# 셀 4: 수용체 준비 (3RPR chain A, 물 217개·리간드 25Z 제거, pH 7.4 수소 추가)
import urllib.request
pdb_path = f'{OUT}/3RPR.pdb'
urllib.request.urlretrieve('https://files.rcsb.org/download/3RPR.pdb', pdb_path)

prot, lig, n_water = [], [], 0
for line in open(pdb_path):
    rec = line[:6].strip()
    if rec == 'ATOM' and line[21] == 'A':
        prot.append(line)
    elif rec == 'HETATM':
        if line[17:20] == 'HOH': n_water += 1
        elif line[17:20] == '25Z' and line[21] == 'A': lig.append(line)

open('/content/receptor_noH.pdb', 'w').writelines(prot + ['END\n'])
open(f'{OUT}/ligand_25Z_crystal.pdb', 'w').writelines(lig + ['END\n'])   # 다음 단계 redocking용
subprocess.run(['obabel', '/content/receptor_noH.pdb', '-O', '/content/receptor.pdb', '-h', '-p', '7.4'], capture_output=True)
if not os.path.exists('/content/receptor.pdb'):   # obabel 실패 시 수소 없이 진행 (GNINA가 자체적으로 처리)
    shutil.copy('/content/receptor_noH.pdb', '/content/receptor.pdb')
shutil.copy('/content/receptor.pdb', f'{OUT}/receptor.pdb')
print(f'단백질 원자 {len(prot)}개, 제거한 물 {n_water}개, 25Z 원자 {len(lig)}개')

# 도킹 박스 (진행 상황 문서에서 확정한 값)
BOX = dict(center_x=92.37, center_y=80.29, center_z=-47.95, size_x=16.1, size_y=15.8, size_z=16.5)

In [ ]:
# 셀 5: 리간드 20개 준비
# 1순위: Drive의 cdk2/experiment/01_chembl_cdk2_activity.csv (노트북에서 업로드해둔 경우)
# 2순위: 없으면 ChEMBL API에서 CDK2(CHEMBL301) 화합물을 직접 받음
import pandas as pd, json
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors

csv = f'{BASE}/experiment/01_chembl_cdk2_activity.csv'
if os.path.exists(csv):
    df = pd.read_csv(csv)
    smi_col = [c for c in df.columns if 'smiles' in c.lower()][0]
    id_col  = [c for c in df.columns if 'chembl' in c.lower() and 'id' in c.lower()]
    df = df.rename(columns={smi_col: 'smiles'})
    df['cid'] = df[id_col[0]] if id_col else [f'mol{i}' for i in range(len(df))]
    src = '01_chembl_cdk2_activity.csv'
else:
    url = ('https://www.ebi.ac.uk/chembl/api/data/activity.json?target_chembl_id=CHEMBL301'
           '&standard_type=IC50&limit=300&only=molecule_chembl_id,canonical_smiles')
    acts = json.load(urllib.request.urlopen(url))['activities']
    df = pd.DataFrame(acts).rename(columns={'canonical_smiles': 'smiles', 'molecule_chembl_id': 'cid'})
    src = 'ChEMBL API (CHEMBL301 IC50)'
df = df.dropna(subset=['smiles']).drop_duplicates('cid')
sample = df.sample(n=20, random_state=42)
print('리간드 출처:', src)

lig_files = []
for cid, smi in zip(sample['cid'], sample['smiles']):
    m = Chem.MolFromSmiles(smi)
    if m is None: continue
    mh = Chem.AddHs(m)
    if AllChem.EmbedMolecule(mh, AllChem.ETKDGv3()) != 0: continue
    AllChem.MMFFOptimizeMolecule(mh)
    mh.SetProp('_Name', str(cid))
    f = f'{OUT}/ligands/{cid}.sdf'
    Chem.SDWriter(f).write(mh)
    lig_files.append((str(cid), f, m.GetNumHeavyAtoms(), Descriptors.NumRotatableBonds(m)))
print(f'3D 구조 생성 완료: {len(lig_files)}개')
pd.DataFrame(lig_files, columns=['cid','file','heavy_atoms','rot_bonds'])[['cid','heavy_atoms','rot_bonds']]

In [ ]:
# 셀 6: 도킹 속도 측정
# A: 기본 설정 (CNN rescore, exhaustiveness 8) — 실제 3,000개 도킹에 쓸 설정, 20개 전부
# B: CNN 끔 (Vina 방식 점수만) — CNN 점수 계산 비용 비교용, 5개
import time, re
from rdkit import Chem

def dock(cid, lig, tag, extra):
    out = f'{OUT}/docked/{cid}_{tag}.sdf'
    cmd = [GNINA, '-r', '/content/receptor.pdb', '-l', lig, '-o', out,
           '--exhaustiveness', '8', '--num_modes', '9', '--seed', '1'] + extra
    for k, v in BOX.items(): cmd += [f'--{k}', str(v)]
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    sec = time.time() - t0
    gpu_warn = 'No GPU' in (r.stdout + r.stderr) or 'CUDA' in r.stderr and 'error' in r.stderr.lower()
    best = {}
    try:
        m = next(iter(Chem.SDMolSupplier(out)))
        for k in ['minimizedAffinity', 'CNNscore', 'CNNaffinity']:
            if m.HasProp(k): best[k] = float(m.GetProp(k))
    except Exception:
        print('  ! 실패:', r.stderr[-300:])
    return dict(cid=cid, config=tag, seconds=round(sec, 1), gpu_warning=gpu_warn, **best)

rows = []
info = {c: (h, rb) for c, _, h, rb in lig_files}
print('A: 기본 설정 (CNN rescore)')
for i, (cid, f, _, _) in enumerate(lig_files, 1):
    r = dock(cid, f, 'A_default', [])
    rows.append(r)
    print(f"  {i:2d}/{len(lig_files)} {cid:15s} {r['seconds']:6.1f}초  Vina {r.get('minimizedAffinity','-')}  CNNscore {r.get('CNNscore','-')}")
print('B: CNN 끔')
for cid, f, _, _ in lig_files[:5]:
    r = dock(cid, f, 'B_no_cnn', ['--cnn_scoring', 'none'])
    rows.append(r)
    print(f"  {cid:15s} {r['seconds']:6.1f}초")

res = pd.DataFrame(rows)
res['heavy_atoms'] = res['cid'].map(lambda c: info[c][0])
res['rot_bonds']   = res['cid'].map(lambda c: info[c][1])
res.to_csv(f'{OUT}/02_gnina_timing.csv', index=False)
if res['gpu_warning'].any():
    print('\n⚠ GPU를 못 쓰고 있다는 경고가 있음 — 런타임 유형이 GPU인지 확인')
print('\n저장:', f'{OUT}/02_gnina_timing.csv')

In [ ]:
# 셀 7: 3,000개 예상 시간 + skill/02_docking.md 에 붙일 요약
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'GPU 없음'
A = res[res.config == 'A_default'].iloc[1:]   # 첫 실행은 모델 로딩 포함이라 제외
B = res[res.config == 'B_no_cnn']
first = res[res.config == 'A_default'].iloc[0]['seconds']
medA, meanA, medB = A.seconds.median(), A.seconds.mean(), B.seconds.median()
hours = lambda s, n=3000: s * n / 3600

summary = f"""## GNINA 도킹 속도 측정 (Colab)

| 항목 | 값 |
|---|---|
| GNINA 버전 | {GNINA_VERSION} |
| GPU | {gpu} |
| 수용체 | PDB 3RPR chain A, 물·리간드 제거, pH 7.4 수소 추가 (OpenBabel) |
| 도킹 박스 | center (92.37, 80.29, -47.95), size (16.1, 15.8, 16.5) Å |
| 설정 | exhaustiveness 8, num_modes 9, seed 1 |
| 측정 리간드 | {len(lig_files)}개 (CDK2 ChEMBL 화합물 무작위, 중원자 {A.heavy_atoms.min()}~{A.heavy_atoms.max()}개) |

| 설정 | 1개당 시간 (중앙값) | 3,000개 예상 |
|---|---|---|
| A. CNN rescore (기본, 채택) | {medA:.1f}초 (평균 {meanA:.1f}초) | {hours(medA):.1f}시간 |
| B. CNN 끔 (Vina 점수만) | {medB:.1f}초 | {hours(medB):.1f}시간 |

- 첫 실행은 CNN 모델 로딩 포함 {first:.1f}초로 계산에서 제외.
- 도킹 점수 (A, Vina affinity): {A.minimizedAffinity.min():.2f} ~ {A.minimizedAffinity.max():.2f} kcal/mol
- 무료 Colab은 세션이 끊기므로 Drive에 결과를 1개씩 바로 저장하며 여러 세션에 나눠 실행.
"""
open(f'{OUT}/02_docking_speed_summary.md', 'w').write(summary)
print(summary)
print('저장:', f'{OUT}/02_docking_speed_summary.md')